# UMUD — Apo Gray55 Bbox Pipeline (Phase 3 v3)

**GPU notebook** — implements the refined context hypothesis:

1. Detect ultrasound ROI bbox (non-black content).
2. Replace everything **outside** the bbox with fixed gray **RGB (55, 55, 55)** → grayscale 55.
3. Run apo inference on the preprocessed image.
4. **Zero predicted mask pixels outside the bbox** before geometry (removes gutter noise).
5. Compare three pipelines on all test images:
   - **Baseline** (raw image, raw pred)
   - **Gray55+bbox** (gray fill + mask clip)
   - **Gray55+stretch+bbox** (gray fill + percentile contrast stretch inside ROI + mask clip)

Outputs:
- `/kaggle/working/apo_contrast_fill_compare.csv`
- Gallery figures under `/kaggle/working/figures/apo_contrast_fill/`



## Configuration


In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import matplotlib.pyplot as plt

IMG_SIZE = 256
APO_REGION_THRESHOLD = 0.50

N_GALLERY_FAIL = 10
N_GALLERY_OK = 6
RANDOM_SEED = 42

ROI_THRESH = 5
ROI_PAD_PX = 10

# Working-cohort gray from visual inspection (RGB 55,55,55 at full opacity)
GRAY_FILL_VALUE = 55

# Contrast stretch inside ROI only (percentile clip then linear rescale to 0..255)
P_LOW = 1
P_HIGH = 99

MASK_OVERLAY_ALPHA = 0.55
APO_OVERLAY_COLOR = (255, 140, 0)

FIG_DIR = Path("/kaggle/working/figures/apo_contrast_fill")
FIG_DIR.mkdir(parents=True, exist_ok=True)

APO_MODEL_PATH = Path(
    "/kaggle/input/notebooks/ucheozoemena/umud-train-apo-mounted-phase-3/apo_baseline.pkl"
)

COMPETITION_DIR = Path(
    "/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data"
)
TEST_DIR = COMPETITION_DIR / "test_images_v2/test_set_v2"


In [ ]:
from __future__ import annotations

import cv2
import numpy as np
import pandas as pd
from fastai.vision.all import PILImage, load_learner
from PIL import Image
from tqdm.auto import tqdm

IMAGE_EXTS = {".tif", ".tiff", ".png", ".jpg", ".jpeg"}


def list_test_images(directory: Path) -> list[Path]:
    files = [
        p
        for p in directory.rglob("*")
        if p.suffix.lower() in IMAGE_EXTS and p.name != "Thumbs.db"
    ]
    # Prefer .tif when both .tif and .png exist for same stem
    by_stem: dict[str, Path] = {}
    for p in sorted(files):
        stem = p.stem
        if stem not in by_stem or p.suffix.lower() in {".tif", ".tiff"}:
            by_stem[stem] = p
    return sorted(by_stem.values(), key=lambda p: p.name)


def load_gray(path: Path) -> np.ndarray:
    with Image.open(path) as im:
        arr = np.array(im)
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    return arr.astype(np.uint8)


def resize_image(img: np.ndarray, size: int) -> np.ndarray:
    return np.array(Image.fromarray(img).resize((size, size), Image.BILINEAR), dtype=np.uint8)


def resize_mask_to(mask: np.ndarray, target_h: int, target_w: int) -> np.ndarray:
    if mask.shape == (target_h, target_w):
        return (mask > 0).astype(np.uint8)
    src = (mask > 0).astype(np.uint8) * 255
    out = Image.fromarray(src).resize((target_w, target_h), Image.NEAREST)
    return (np.array(out) > 0).astype(np.uint8)


def tensor_to_mask(pred) -> np.ndarray:
    if hasattr(pred, "cpu"):
        pred = pred.cpu().numpy()
    arr = np.asarray(pred)
    if arr.ndim == 3:
        arr = arr.argmax(axis=0)
    return (arr > 0).astype(np.uint8)


def open_rgb_256(img_native: np.ndarray) -> PILImage:
    small = resize_image(img_native, IMG_SIZE)
    rgb = np.stack([small, small, small], axis=-1).astype(np.uint8)
    return PILImage.create(rgb)


def tag_apo_style(coverage: float) -> str:
    return "region" if coverage >= APO_REGION_THRESHOLD else "line"


def invert_mask(mask: np.ndarray) -> np.ndarray:
    return (1 - mask).astype(np.uint8)


def effective_apo_mask(mask: np.ndarray, style: str) -> tuple[np.ndarray, str]:
    if style == "region":
        return invert_mask(mask), "inverted_region"
    return mask, "raw_line"


def find_apo_contours(mask: np.ndarray, min_area_frac: float = 0.0003) -> list[np.ndarray]:
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    min_area = mask.size * min_area_frac
    big = [c for c in contours if cv2.contourArea(c) >= min_area]
    big.sort(key=lambda c: cv2.boundingRect(c)[1])
    return big


def pick_superficial_deep(contours: list[np.ndarray], min_sep_px: int = 15):
    if len(contours) < 2:
        return None, None, len(contours)
    sup = contours[0]
    _, y0, _, _ = cv2.boundingRect(sup)
    deep = None
    for c in contours[1:]:
        _, y1, _, _ = cv2.boundingRect(c)
        if y1 >= y0 + min_sep_px:
            deep = c
            break
    if deep is None:
        deep = contours[min(2, len(contours) - 1)]
    return sup, deep, len(contours)


def edge_polyline(contour: np.ndarray, which: str = "bottom", n_bins: int = 60):
    pts = contour.reshape(-1, 2)
    if len(pts) < 3:
        return np.array([]), np.array([])
    x_min, x_max = pts[:, 0].min(), pts[:, 0].max()
    if x_max <= x_min:
        return pts[:, 0].astype(float), pts[:, 1].astype(float)
    edges = np.linspace(x_min, x_max, n_bins + 1)
    xs_out, ys_out = [], []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        in_bin = pts[(pts[:, 0] >= lo) & (pts[:, 0] < hi)]
        if len(in_bin) == 0:
            continue
        y = in_bin[:, 1].max() if which == "bottom" else in_bin[:, 1].min()
        xs_out.append((lo + hi) / 2.0)
        ys_out.append(float(y))
    return np.array(xs_out), np.array(ys_out)


def fit_line(xs: np.ndarray, ys: np.ndarray):
    if len(xs) < 2:
        return None
    return np.poly1d(np.polyfit(xs, ys, 1))


def line_angle_deg(line) -> float:
    return float(np.degrees(np.arctan(line[1]))) if line is not None else np.nan


def mt_from_apo_edges(sup_line, deep_line, x_left: float, x_right: float) -> float:
    if sup_line is None or deep_line is None or x_right <= x_left:
        return np.nan
    thirds = [x_left + (x_right - x_left) * t for t in (1 / 6, 3 / 6, 5 / 6)]
    dists = [abs(float(deep_line(x) - sup_line(x))) for x in thirds]
    return float(np.mean(dists))


def fascicle_pca(mask: np.ndarray) -> dict | None:
    ys, xs = np.where(mask > 0)
    if len(xs) < 3:
        return None
    coords = np.column_stack([xs.astype(float), ys.astype(float)])
    centered = coords - coords.mean(axis=0)
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    direction = vh[0]
    projections = centered @ direction
    return {
        "length_px": float(projections.max() - projections.min()),
        "angle_deg": float(np.degrees(np.arctan2(direction[1], direction[0]))),
    }


def acute_angle_deg(a1: float, a2: float) -> float:
    d = abs(a1 - a2) % 180.0
    return float(min(d, 180.0 - d))


def apo_geometry_attempt(apo_mask: np.ndarray, style: str) -> dict:
    eff, method = effective_apo_mask(apo_mask, style)
    contours = find_apo_contours(eff)
    sup_c, deep_c, n_contours = pick_superficial_deep(contours)
    out = {
        "apo_method": method,
        "n_contours": n_contours,
        "mt_px": np.nan,
        "deep_angle_deg": np.nan,
        "mt_fail_reason": "ok",
        "sup_line": None,
        "deep_line": None,
        "sup_xs": None,
        "sup_ys": None,
        "deep_xs": None,
        "deep_ys": None,
    }
    if len(contours) == 0:
        out["mt_fail_reason"] = "no_contours"
        return out
    if n_contours < 2 or sup_c is None or deep_c is None:
        out["mt_fail_reason"] = "single_contour"
        return out
    sup_x, sup_y = edge_polyline(sup_c, which="bottom")
    deep_x, deep_y = edge_polyline(deep_c, which="top")
    sup_line = fit_line(sup_x, sup_y)
    deep_line = fit_line(deep_x, deep_y)
    out.update(sup_line=sup_line, deep_line=deep_line, sup_xs=sup_x, sup_ys=sup_y, deep_xs=deep_x, deep_ys=deep_y)
    if sup_line is None or deep_line is None:
        out["mt_fail_reason"] = "line_fit_fail"
        return out
    if len(sup_x) == 0 or len(deep_x) == 0:
        out["mt_fail_reason"] = "empty_edge_polyline"
        return out
    x_left = max(sup_x.min(), deep_x.min())
    x_right = min(sup_x.max(), deep_x.max())
    if x_right <= x_left:
        out["mt_fail_reason"] = "no_x_overlap"
        return out
    out["mt_px"] = mt_from_apo_edges(sup_line, deep_line, x_left, x_right)
    out["deep_angle_deg"] = line_angle_deg(deep_line)
    if np.isnan(out["mt_px"]):
        out["mt_fail_reason"] = "mt_compute_nan"
    return out


def apo_geometry_from_mask(apo_mask: np.ndarray, style: str) -> dict:
    primary = apo_geometry_attempt(apo_mask, style)
    primary["geometry_path"] = primary["apo_method"]
    if not np.isnan(primary["mt_px"]):
        return primary
    if style == "region":
        fallback = apo_geometry_attempt(apo_mask, "line")
        if not np.isnan(fallback["mt_px"]):
            fallback["apo_method"] = f"{primary['apo_method']}+fallback_line"
            fallback["geometry_path"] = "fallback_line"
            fallback["mt_fail_reason_primary"] = primary["mt_fail_reason"]
            return fallback
    primary["mt_fail_reason_primary"] = primary["mt_fail_reason"]
    return primary


def derive_geometry(fasc_mask: np.ndarray, apo_mask: np.ndarray, apo_style: str) -> dict:
    apo = apo_geometry_from_mask(apo_mask, apo_style)
    fpca = fascicle_pca(fasc_mask)
    out = {
        "pa_deg": np.nan,
        "fl_px": np.nan,
        "mt_px": apo["mt_px"],
        "apo_method": apo.get("apo_method"),
        "geometry_path": apo.get("geometry_path"),
        "n_contours": apo.get("n_contours"),
        "mt_fail_reason": apo.get("mt_fail_reason"),
        "mt_fail_reason_primary": apo.get("mt_fail_reason_primary"),
        "apo_cov": float(apo_mask.mean()),
        "apo_fg_pixels": int(apo_mask.sum()),
        "fasc_cov": float(fasc_mask.mean()),
        "fasc_fg_pixels": int(fasc_mask.sum()),
    }
    if fpca is not None:
        out["fl_px"] = fpca["length_px"]
        ref = apo["deep_angle_deg"] if not np.isnan(apo["deep_angle_deg"]) else 0.0
        out["pa_deg"] = acute_angle_deg(fpca["angle_deg"], ref)
    return out


In [ ]:
def overlay(img_gray: np.ndarray, mask: np.ndarray, color=APO_OVERLAY_COLOR, alpha=MASK_OVERLAY_ALPHA):
    rgb = np.stack([img_gray, img_gray, img_gray], axis=-1).astype(np.float32)
    tint = np.zeros_like(rgb)
    tint[..., 0], tint[..., 1], tint[..., 2] = color
    sel = mask.astype(bool)
    rgb[sel] = (1 - alpha) * rgb[sel] + alpha * tint[sel]
    return rgb.astype(np.uint8)


def find_roi_bbox(img_gray: np.ndarray, thr: float = ROI_THRESH, pad: int = ROI_PAD_PX):
    import cv2

    roi = (img_gray > thr).astype(np.uint8)
    if roi.sum() == 0:
        h, w = img_gray.shape
        return 0, h, 0, w

    num, labels, stats, _ = cv2.connectedComponentsWithStats(roi, connectivity=8)
    best = max(range(1, num), key=lambda i: stats[i, cv2.CC_STAT_AREA])
    x, y, w, h = stats[best, :4]

    y0 = max(0, y - pad)
    y1 = min(img_gray.shape[0], y + h + pad)
    x0 = max(0, x - pad)
    x1 = min(img_gray.shape[1], x + w + pad)
    return int(y0), int(y1), int(x0), int(x1)


def clip_mask_to_bbox(mask: np.ndarray, bbox: tuple[int, int, int, int]) -> np.ndarray:
    y0, y1, x0, x1 = bbox
    out = np.zeros_like(mask, dtype=np.uint8)
    out[y0:y1, x0:x1] = mask[y0:y1, x0:x1]
    return out


def contrast_stretch_roi(img_gray: np.ndarray, bbox: tuple[int, int, int, int]) -> np.ndarray:
    y0, y1, x0, x1 = bbox
    out = img_gray.astype(np.float32).copy()
    roi = out[y0:y1, x0:x1]
    lo = np.percentile(roi, P_LOW)
    hi = np.percentile(roi, P_HIGH)
    if hi <= lo + 1e-6:
        return img_gray
    roi_clipped = np.clip(roi, lo, hi)
    out[y0:y1, x0:x1] = (roi_clipped - lo) / (hi - lo) * 255.0
    return out.astype(np.uint8)


def preprocess_gray55(img_native: np.ndarray, do_stretch: bool = False):
    """Gray-fill outside bbox with fixed 55; optional contrast stretch inside ROI."""
    h, w = img_native.shape
    bbox = find_roi_bbox(img_native)
    y0, y1, x0, x1 = bbox

    pre = img_native.copy()
    outside = np.ones((h, w), dtype=bool)
    outside[y0:y1, x0:x1] = False
    pre[outside] = GRAY_FILL_VALUE

    if do_stretch:
        pre = contrast_stretch_roi(pre, bbox)

    return pre, bbox


def infer_apo_on_image(img_gray: np.ndarray, bbox: tuple[int, int, int, int], clip_bbox: bool):
    h, w = img_gray.shape
    pil = open_rgb_256(img_gray)
    _, apo_t, _ = apo_learn.predict(pil)
    mask = resize_mask_to(tensor_to_mask(apo_t), h, w)
    if clip_bbox:
        mask = clip_mask_to_bbox(mask, bbox)
    style = tag_apo_style(float(mask.mean()))
    geo = apo_geometry_from_mask(mask, style)
    return mask, style, geo


In [ ]:
assert APO_MODEL_PATH.exists(), f"Missing apo model: {APO_MODEL_PATH}"
assert TEST_DIR.exists(), f"Missing test dir: {TEST_DIR}"

apo_learn = load_learner(APO_MODEL_PATH)
test_paths = list_test_images(TEST_DIR)
print(f"Test images: {len(test_paths)}")


In [ ]:
rows = []

for path in tqdm(test_paths, desc="compare pipelines"):
    img_native = load_gray(path)
    h, w = img_native.shape

    # Baseline
    bbox = find_roi_bbox(img_native)
    base_mask, base_style, base_geo = infer_apo_on_image(img_native, bbox, clip_bbox=False)

    # Gray55 + bbox mask clip
    img_g, bbox_g = preprocess_gray55(img_native, do_stretch=False)
    g_mask, g_style, g_geo = infer_apo_on_image(img_g, bbox_g, clip_bbox=True)

    # Gray55 + stretch + bbox mask clip
    img_s, bbox_s = preprocess_gray55(img_native, do_stretch=True)
    s_mask, s_style, s_geo = infer_apo_on_image(img_s, bbox_s, clip_bbox=True)

    rows.append(
        {
            "image_id": path.name,
            "res": f"{h}x{w}",
            "bbox": f"{bbox_g[0]}:{bbox_g[1]}:{bbox_g[2]}:{bbox_g[3]}",
            "base_pred_cov": float(base_mask.mean()),
            "gray_pred_cov": float(g_mask.mean()),
            "stretch_pred_cov": float(s_mask.mean()),
            "base_style": base_style,
            "gray_style": g_style,
            "stretch_style": s_style,
            "base_mt_ok": bool(not np.isnan(base_geo["mt_px"])),
            "gray_mt_ok": bool(not np.isnan(g_geo["mt_px"])),
            "stretch_mt_ok": bool(not np.isnan(s_geo["mt_px"])),
            "base_mt_fail_reason": base_geo["mt_fail_reason"],
            "gray_mt_fail_reason": g_geo["mt_fail_reason"],
            "stretch_mt_fail_reason": s_geo["mt_fail_reason"],
            "base_mt_px": float(base_geo["mt_px"]) if not np.isnan(base_geo["mt_px"]) else np.nan,
            "gray_mt_px": float(g_geo["mt_px"]) if not np.isnan(g_geo["mt_px"]) else np.nan,
            "stretch_mt_px": float(s_geo["mt_px"]) if not np.isnan(s_geo["mt_px"]) else np.nan,
        }
    )

df = pd.DataFrame(rows)
out_csv = "/kaggle/working/apo_contrast_fill_compare.csv"
df.to_csv(out_csv, index=False)
print("Wrote:", out_csv)

print("\n=== MT OK rates ===")
print("baseline:", float(df.base_mt_ok.mean()))
print("gray55+bbox:", float(df.gray_mt_ok.mean()))
print("gray55+stretch+bbox:", float(df.stretch_mt_ok.mean()))

print("\nbaseline fail:", df.loc[~df.base_mt_ok, "base_mt_fail_reason"].value_counts().to_dict())
print("gray55+bbox fail:", df.loc[~df.gray_mt_ok, "gray_mt_fail_reason"].value_counts().to_dict())
print("stretch+bbox fail:", df.loc[~df.stretch_mt_ok, "stretch_mt_fail_reason"].value_counts().to_dict())

print("\nMT-fixed vs baseline:")
print("gray55+bbox:", int(((~df.base_mt_ok) & (df.gray_mt_ok)).sum()))
print("stretch+bbox:", int(((~df.base_mt_ok) & (df.stretch_mt_ok)).sum()))
base_no = df[df.base_mt_fail_reason == "no_contours"]
print("no_contours fixed by gray55+bbox:", int(((~base_no.base_mt_ok) & (base_no.gray_mt_ok)).sum()), "/", len(base_no))
print("no_contours fixed by stretch+bbox:", int(((~base_no.base_mt_ok) & (base_no.stretch_mt_ok)).sum()), "/", len(base_no))


In [ ]:
fail_cases = df[df.base_mt_fail_reason == "no_contours"].copy()
ok_cases = df[df.base_mt_ok].copy()

fail_pick = fail_cases.sample(min(N_GALLERY_FAIL, len(fail_cases)), random_state=RANDOM_SEED) if len(fail_cases) else fail_cases
ok_pick = ok_cases.sample(min(N_GALLERY_OK, len(ok_cases)), random_state=RANDOM_SEED) if len(ok_cases) else ok_cases


def show_one(image_id: str):
    path = TEST_DIR / image_id
    img_native = load_gray(path)
    bbox = find_roi_bbox(img_native)

    img_g, _ = preprocess_gray55(img_native, do_stretch=False)
    img_s, _ = preprocess_gray55(img_native, do_stretch=True)

    base_mask, _, base_geo = infer_apo_on_image(img_native, bbox, clip_bbox=False)
    g_mask, _, g_geo = infer_apo_on_image(img_g, bbox, clip_bbox=True)
    s_mask, _, s_geo = infer_apo_on_image(img_s, bbox, clip_bbox=True)

    # noise outside bbox on raw pred
    outside_noise = int(base_mask.sum() - clip_mask_to_bbox(base_mask, bbox).sum())

    fig, axes = plt.subplots(2, 5, figsize=(28, 9))

    # Row 0: images + bbox
    axes[0, 0].imshow(img_native, cmap="gray")
    axes[0, 0].set_title("raw", fontsize=9)
    axes[0, 0].axis("off")

    axes[0, 1].imshow(img_g, cmap="gray", vmin=0, vmax=255)
    axes[0, 1].set_title(f"gray55 outside bbox\nfill={GRAY_FILL_VALUE}", fontsize=9)
    axes[0, 1].axis("off")

    axes[0, 2].imshow(img_s, cmap="gray", vmin=0, vmax=255)
    axes[0, 2].set_title("gray55 + stretch", fontsize=9)
    axes[0, 2].axis("off")

    for ax, img, title in zip(axes[0, 3:], [img_native, img_g], ["bbox on raw", "bbox on gray55"]):
        ax.imshow(img, cmap="gray")
        y0, y1, x0, x1 = bbox
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="cyan", linewidth=2))
        ax.set_title(title, fontsize=9)
        ax.axis("off")

    # Row 1: preds
    axes[1, 0].imshow(base_mask, cmap="gray", vmin=0, vmax=1)
    axes[1, 0].set_title(f"base pred\n{base_geo['mt_fail_reason']}", fontsize=9)
    axes[1, 0].axis("off")

    axes[1, 1].imshow(g_mask, cmap="gray", vmin=0, vmax=1)
    axes[1, 1].set_title(f"gray55+bbox\n{g_geo['mt_fail_reason']}", fontsize=9)
    axes[1, 1].axis("off")

    axes[1, 2].imshow(s_mask, cmap="gray", vmin=0, vmax=1)
    axes[1, 2].set_title(f"stretch+bbox\n{s_geo['mt_fail_reason']}", fontsize=9)
    axes[1, 2].axis("off")

    axes[1, 3].imshow(overlay(img_native, base_mask))
    axes[1, 3].set_title(f"base ov\noutside noise px={outside_noise}", fontsize=9)
    axes[1, 3].axis("off")

    axes[1, 4].imshow(overlay(img_g, g_mask))
    axes[1, 4].set_title("gray55 ov", fontsize=9)
    axes[1, 4].axis("off")

    plt.suptitle(
        f"{image_id} | base mt={base_geo['mt_px'] if not np.isnan(base_geo['mt_px']) else 'NaN'} "
        f"| gray mt={g_geo['mt_px'] if not np.isnan(g_geo['mt_px']) else 'NaN'} "
        f"| stretch mt={s_geo['mt_px'] if not np.isnan(s_geo['mt_px']) else 'NaN'}",
        y=1.02,
        fontsize=11,
    )
    plt.tight_layout()
    return fig


print("\n=== Gallery: baseline no_contours ===")
for i, image_id in enumerate(fail_pick.image_id.tolist(), start=1):
    print(f"[fail {i}] {image_id}")
    fig = show_one(image_id)
    out = FIG_DIR / f"gallery_fail_{i:02d}_{Path(image_id).stem}.png"
    fig.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("\n=== Gallery: baseline MT OK ===")
for i, image_id in enumerate(ok_pick.image_id.tolist(), start=1):
    print(f"[ok {i}] {image_id}")
    fig = show_one(image_id)
    out = FIG_DIR / f"gallery_ok_{i:02d}_{Path(image_id).stem}.png"
    fig.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("Done. Figures in:", FIG_DIR)
